# Official DEAL held-out denoising benchmark

This notebook reproduces the CSV used for the official pretrained DEAL
comparison. It evaluates **only DEAL**; the project-trained U-TV,
U-TGV, and U-Tikhonov models are evaluated in Notebooks 6b and 8.

The protocol is deliberately identical to the main held-out benchmark:

- all 100 DIV2K validation images and five geometric phantoms;
- grayscale images of size $256\times256$;
- Gaussian noise levels $\sigma\in\{0.05,0.10,0.15,0.20\}$;
- paired deterministic noise across methods and noise levels;
- the authors' released grayscale DEAL checkpoint, evaluated with the
  official stopping criteria.

The principal output is `per_image_metrics_deal_official.csv`. Notebook
6 discovers this file, verifies its sample hashes against its own
manifest, and combines it with the project-model results. DEAL is an
external pretrained reference: its training data and optimisation
budget are not controlled against the project models.


In [1]:
from __future__ import annotations

import os
import sys
from pathlib import Path


def locate_project_root() -> Path:
    '''Find cleaned_pipeline regardless of where Jupyter was launched.'''
    cwd = Path.cwd().resolve()
    candidates = []
    for base in (cwd, *cwd.parents):
        candidates.extend((base, base / "cleaned_pipeline"))
    for candidate in candidates:
        if (candidate / "src").is_dir() and (candidate / "scripts").is_dir():
            return candidate.resolve()
    raise FileNotFoundError(
        "Could not find cleaned_pipeline. Start Jupyter from Thesis/Python, "
        "cleaned_pipeline, or cleaned_pipeline/notebooks."
    )


PROJECT_ROOT = locate_project_root()
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)


Project root: /Users/kyrillguba/Documents/Data Analytics MSc/Thesis/Python/cleaned_pipeline


In [2]:
import gc
import inspect
import time

import numpy as np
import pandas as pd
import torch
from torch.utils.data import ConcatDataset, DataLoader, Subset

from src.data.benchmark_dataset import (
    GeometricShapesDataset,
    MultiNoiseBenchmarkDataset,
    benchmark_manifest,
)
from src.data.div2k_dataset import DIV2KDenoisingDataset
from src.evaluation.benchmark import (
    benchmark_inference_runtime,
    method_summary,
    runtime_summary,
)
from src.evaluation.denoising_eval import evaluate_denoising_model
from src.models.deal_official import load_official_deal_model
from src.training.denoising import get_device

device = get_device()
print("PyTorch:", torch.__version__)
print("Device:", device)


PyTorch: 2.12.0
Device: mps


## Configuration

`RECOMPUTE_DEAL=False` reuses a compatible cached DEAL CSV when it is
present. If the CSV does not exist, the benchmark is run automatically.
Set it to `True` to force a complete re-evaluation. The official solver
is iterative and the full 420-sample run can take several hours.


In [3]:
FIRST_TRY_ROOT = PROJECT_ROOT.parent / "First Try"


def locate_div2k_validation_root() -> Path:
    likely_paths = [
        FIRST_TRY_ROOT / "DIV2K" / "DIV2K_valid_HR",
        FIRST_TRY_ROOT / "Datasets" / "DIV2K" / "DIV2K_valid_HR",
        FIRST_TRY_ROOT / "data" / "DIV2K" / "DIV2K_valid_HR",
        Path.home() / "Datasets" / "DIV2K" / "DIV2K_valid_HR",
    ]
    for path in likely_paths:
        if path.is_dir():
            return path.resolve()
    if FIRST_TRY_ROOT.is_dir():
        matches = sorted(
            path.resolve()
            for path in FIRST_TRY_ROOT.rglob("DIV2K_valid_HR")
            if path.is_dir()
        )
        if len(matches) == 1:
            return matches[0]
        if len(matches) > 1:
            raise RuntimeError(
                "Several DIV2K_valid_HR folders were found. "
                f"Set DIV2K_ROOT manually: {matches}"
            )
    raise FileNotFoundError(
        "DIV2K_valid_HR was not found. Set DIV2K_ROOT manually."
    )


DIV2K_ROOT = locate_div2k_validation_root()
DEAL_METHOD = "DEAL (official pretrained)"
DEAL_CHECKPOINT = (
    PROJECT_ROOT / "checkpoints" / "deal_official" / "deal_gray.pth"
)
DEAL_PROTOCOL = "official_notebook"

IMAGE_SIZE = 256
SIGMAS = (0.05, 0.10, 0.15, 0.20)
TEST_NOISE_SEED = 20_000
BATCH_SIZE = 1
NUM_WORKERS = 0

RECOMPUTE_DEAL = False
PROGRESS_EVERY = 10

# Matches the disclosed timing protocol used in the thesis comparison.
TIMING_SIGMA = 0.10
TIMING_MAX_BATCHES = 3
TIMING_WARMUP_RUNS = 1
TIMING_REPEATS = 3

OUTPUT_DIR = PROJECT_ROOT / "results" / "deal_official_benchmark"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DEAL_RESULTS_PATH = OUTPUT_DIR / "per_image_metrics_deal_official.csv"

print("DIV2K:", DIV2K_ROOT)
print("Official DEAL checkpoint:", DEAL_CHECKPOINT)
print("Official DEAL protocol:", DEAL_PROTOCOL)
print("Output:", OUTPUT_DIR)


DIV2K: /Users/kyrillguba/Documents/Data Analytics MSc/Thesis/Python/First Try/data/DIV2K/DIV2K_valid_HR
Official DEAL checkpoint: /Users/kyrillguba/Documents/Data Analytics MSc/Thesis/Python/cleaned_pipeline/checkpoints/deal_official/deal_gray.pth
Official DEAL protocol: official_notebook
Output: /Users/kyrillguba/Documents/Data Analytics MSc/Thesis/Python/cleaned_pipeline/results/deal_official_benchmark


## Recreate and verify the exact paired benchmark

`MultiNoiseBenchmarkDataset` uses a local random generator for each
image. The same standard-normal draw is scaled at the four noise levels,
so comparisons across $\sigma$ are paired. The manifest records the
sample ID, seed, and SHA-256 hashes of both clean and noisy tensors.


In [4]:
IDENTITY_COLUMNS = [
    "sample_id",
    "dataset",
    "image_name",
    "sigma",
    "noise_seed",
    "clean_sha256",
    "noisy_sha256",
]


def compare_identity(current, reference, reference_path):
    '''Compare sample identities by ID rather than CSV row order.'''
    missing = [
        column for column in IDENTITY_COLUMNS
        if column not in reference.columns
    ]
    if missing:
        return {
            "reference": str(reference_path),
            "exact_match": False,
            "rows": len(reference),
            "problem": f"missing columns: {missing}",
        }

    left = current[IDENTITY_COLUMNS].copy()
    right = reference[IDENTITY_COLUMNS].copy()
    left["sample_id"] = left["sample_id"].astype(str)
    right["sample_id"] = right["sample_id"].astype(str)

    if left["sample_id"].duplicated().any():
        raise RuntimeError("The new manifest contains duplicate sample IDs.")
    if right["sample_id"].duplicated().any():
        return {
            "reference": str(reference_path),
            "exact_match": False,
            "rows": len(reference),
            "problem": "duplicate sample IDs",
        }

    left_ids = set(left["sample_id"])
    right_ids = set(right["sample_id"])
    if len(left) != len(right) or left_ids != right_ids:
        return {
            "reference": str(reference_path),
            "exact_match": False,
            "rows": len(reference),
            "problem": (
                f"sample-ID difference: new-only={len(left_ids - right_ids)}, "
                f"old-only={len(right_ids - left_ids)}"
            ),
        }

    merged = left.merge(
        right,
        on="sample_id",
        suffixes=("_new", "_old"),
        validate="one_to_one",
    )
    mismatch_columns = []
    for column in [
        "dataset", "image_name", "clean_sha256", "noisy_sha256"
    ]:
        different = (
            merged[f"{column}_new"].astype(str)
            != merged[f"{column}_old"].astype(str)
        )
        if different.any():
            mismatch_columns.append(f"{column}={int(different.sum())}")

    sigma_equal = np.isclose(
        pd.to_numeric(merged["sigma_new"]).to_numpy(float),
        pd.to_numeric(merged["sigma_old"]).to_numpy(float),
        rtol=0.0,
        atol=1e-7,
    )
    if not sigma_equal.all():
        mismatch_columns.append(f"sigma={int((~sigma_equal).sum())}")

    seed_equal = (
        pd.to_numeric(merged["noise_seed_new"]).to_numpy(np.int64)
        == pd.to_numeric(merged["noise_seed_old"]).to_numpy(np.int64)
    )
    if not seed_equal.all():
        mismatch_columns.append(f"noise_seed={int((~seed_equal).sum())}")

    return {
        "reference": str(reference_path),
        "exact_match": not mismatch_columns,
        "rows": len(reference),
        "problem": ", ".join(mismatch_columns),
    }


div2k_clean = DIV2KDenoisingDataset(
    root_dir=DIV2K_ROOT,
    image_size=IMAGE_SIZE,
    sigma_min=0.0,
    sigma_max=0.0,
    max_images=None,
    seed=42,
)
if len(div2k_clean) != 100:
    raise RuntimeError(
        f"Expected all 100 DIV2K validation images, found {len(div2k_clean)}."
    )

geometric_clean = GeometricShapesDataset(image_size=IMAGE_SIZE)
div2k_benchmark = MultiNoiseBenchmarkDataset(
    div2k_clean,
    sigmas=SIGMAS,
    seed=TEST_NOISE_SEED,
    dataset_name="div2k",
    paired_noise_across_sigmas=True,
)
geometric_benchmark = MultiNoiseBenchmarkDataset(
    geometric_clean,
    sigmas=SIGMAS,
    seed=TEST_NOISE_SEED + 100_000,
    dataset_name="geometry",
    paired_noise_across_sigmas=True,
)
benchmark_dataset = ConcatDataset([div2k_benchmark, geometric_benchmark])
benchmark_loader = DataLoader(
    benchmark_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
)

manifest = pd.concat(
    [benchmark_manifest(div2k_benchmark), benchmark_manifest(geometric_benchmark)],
    ignore_index=True,
)
if len(manifest) != 420 or manifest["sample_id"].duplicated().any():
    raise RuntimeError("Expected a manifest of 420 unique benchmark samples.")

current_manifest_path = (OUTPUT_DIR / "benchmark_manifest.csv").resolve()
reference_paths = []
results_root = PROJECT_ROOT / "results"
if results_root.is_dir():
    for path in sorted(results_root.rglob("benchmark_manifest.csv")):
        resolved = path.resolve()
        if resolved != current_manifest_path and resolved not in reference_paths:
            reference_paths.append(resolved)

reports = []
exact_matches = []
for path in reference_paths:
    report = compare_identity(manifest, pd.read_csv(path), path)
    reports.append(report)
    if report["exact_match"]:
        exact_matches.append(path)

if reports:
    display(pd.DataFrame(reports))
if exact_matches:
    print("Exact sample IDs, seeds, and tensor hashes match:", exact_matches[0])
elif reference_paths:
    raise RuntimeError(
        "No previous benchmark manifest matches the recreated samples. "
        "Do not evaluate DEAL until the mismatch is resolved."
    )
else:
    print(
        "No earlier manifest was found. The deterministic manifest was "
        "created, but cross-run identity could not be checked."
    )

manifest.to_csv(current_manifest_path, index=False)
display(manifest.groupby(["dataset", "sigma"]).size().rename("images"))
print("Saved:", current_manifest_path)


,reference,exact_match,rows,problem
0,/Users/kyrillguba/Documents/Data Analytics MSc...,True,420,
1,/Users/kyrillguba/Documents/Data Analytics MSc...,True,420,
2,/Users/kyrillguba/Documents/Data Analytics MSc...,True,420,
3,/Users/kyrillguba/Documents/Data Analytics MSc...,True,420,
4,/Users/kyrillguba/Documents/Data Analytics MSc...,True,420,


Exact sample IDs, seeds, and tensor hashes match: /Users/kyrillguba/Documents/Data Analytics MSc/Thesis/Python/cleaned_pipeline/results/benchmark_final_frozen/benchmark_manifest.csv


dataset   sigma
div2k     0.05     100
          0.10     100
          0.15     100
          0.20     100
geometry  0.05       5
          0.10       5
          0.15       5
          0.20       5
Name: images, dtype: int64

Saved: /Users/kyrillguba/Documents/Data Analytics MSc/Thesis/Python/cleaned_pipeline/results/deal_official_benchmark/benchmark_manifest.csv


## Load the official pretrained DEAL model

The loader verifies the official checkpoint and constructs the released
architecture. Known benchmark noise levels are converted from the
normalised image scale to the model's 0-255 convention.


In [5]:
if not DEAL_CHECKPOINT.is_file():
    raise FileNotFoundError(
        f"Official DEAL checkpoint was not found: {DEAL_CHECKPOINT}"
    )

deal_signature = inspect.signature(load_official_deal_model)
deal_kwargs = {}
if "device" in deal_signature.parameters:
    deal_kwargs["device"] = device
if "protocol" in deal_signature.parameters:
    deal_kwargs["protocol"] = DEAL_PROTOCOL

loaded_deal = load_official_deal_model(DEAL_CHECKPOINT, **deal_kwargs)
if isinstance(loaded_deal, tuple):
    if len(loaded_deal) != 2:
        raise RuntimeError(
            "Unexpected official DEAL loader return value: "
            f"tuple length {len(loaded_deal)}."
        )
    deal_model, deal_metadata = loaded_deal
else:
    deal_model = loaded_deal
    deal_metadata = {}

if not isinstance(deal_model, torch.nn.Module):
    raise TypeError("The official DEAL loader did not return a torch.nn.Module.")

deal_model.to(device).eval()
if not bool(getattr(deal_model, "requires_noise_level", True)):
    raise RuntimeError("The loaded DEAL wrapper must receive the known noise level.")

configuration = {
    "method": DEAL_METHOD,
    "comparison_role": "external official pretrained reference",
    "checkpoint_path": str(DEAL_CHECKPOINT),
    "parameter_count": sum(p.numel() for p in deal_model.parameters()),
    "protocol": DEAL_PROTOCOL,
    "repository": deal_metadata.get("repository", "https://github.com/mehrsapo/DEAL"),
    "repository_commit": deal_metadata.get(
        "repository_commit", "554820ca356ff8a78bc49097e3ffcba3875a3ac2"
    ),
    "checkpoint_sha256": deal_metadata.get(
        "checkpoint_sha256",
        "ed3fc0b3284b4951dafb810e8a383e26d543ad1cb8cbfc41484e4f16b847d385",
    ),
    "max_outer_iterations": getattr(deal_model, "max_outer_iterations", np.nan),
    "max_inner_iterations": getattr(deal_model, "max_inner_iterations", np.nan),
    "outer_tolerance": getattr(deal_model, "outer_tolerance", np.nan),
    "inner_tolerance": getattr(deal_model, "inner_tolerance", np.nan),
    "noise_scale_conversion": "sigma_255 = 255 * sigma",
}
model_configuration = pd.DataFrame([configuration])
model_configuration.to_csv(OUTPUT_DIR / "model_configuration.csv", index=False)
display(model_configuration.T.rename(columns={0: "value"}))


,value
method,DEAL (official pretrained)
comparison_role,external official pretrained reference
checkpoint_path,/Users/kyrillguba/Documents/Data Analytics MSc...
parameter_count,468570
protocol,official_notebook
repository,https://github.com/mehrsapo/DEAL
repository_commit,554820ca356ff8a78bc49097e3ffcba3875a3ac2
checkpoint_sha256,ed3fc0b3284b4951dafb810e8a383e26d543ad1cb8cbfc...
max_outer_iterations,1000
max_inner_iterations,1000


## Evaluate DEAL or validate the cached result

The cache is accepted only when its 420 sample IDs, seeds, image names,
noise levels, and tensor hashes match the newly constructed manifest.
Progress is printed during a fresh run so that a long official-solver
evaluation does not appear inactive.


In [6]:
def progress_batches(loader, every=10):
    total = len(loader)
    started = time.perf_counter()
    for index, batch in enumerate(loader, start=1):
        if index == 1 or index % every == 0 or index == total:
            elapsed = time.perf_counter() - started
            rate = elapsed / max(index - 1, 1)
            remaining = rate * max(total - index, 0)
            print(
                f"DEAL sample {index:>3}/{total} | "
                f"elapsed {elapsed / 60:.1f} min | "
                f"estimated remaining {remaining / 60:.1f} min",
                flush=True,
            )
        yield batch


def canonicalize_deal_results(results):
    missing = [column for column in IDENTITY_COLUMNS if column not in results]
    if missing:
        raise RuntimeError(f"DEAL results lack identity columns: {missing}")
    if "method" not in results:
        raise RuntimeError("DEAL results lack the method column.")
    if set(results["method"].astype(str)) != {DEAL_METHOD}:
        raise RuntimeError(
            "The cached CSV is not a DEAL-only result with the expected label."
        )

    report = compare_identity(manifest, results, DEAL_RESULTS_PATH)
    if not report["exact_match"]:
        raise RuntimeError(
            "Cached DEAL results do not match the benchmark manifest: "
            f"{report['problem']}"
        )
    if len(results) != 420 or results["sample_id"].astype(str).duplicated().any():
        raise RuntimeError("Expected exactly 420 unique DEAL result rows.")

    ordered = results.copy()
    ordered["sample_id"] = ordered["sample_id"].astype(str)
    expected_ids = manifest["sample_id"].astype(str).tolist()
    ordered = (
        ordered.set_index("sample_id", drop=False)
        .loc[expected_ids]
        .reset_index(drop=True)
    )
    return ordered


if DEAL_RESULTS_PATH.is_file() and not RECOMPUTE_DEAL:
    deal_results = canonicalize_deal_results(pd.read_csv(DEAL_RESULTS_PATH))
    print("Reusing verified DEAL results:", DEAL_RESULTS_PATH)
else:
    print("Evaluating official DEAL on all 420 paired benchmark samples.")
    deal_results = evaluate_denoising_model(
        deal_model,
        progress_batches(benchmark_loader, every=PROGRESS_EVERY),
        device,
        model_type="deal_official",
        clamp_for_metrics=True,
    )
    deal_results.insert(0, "method", DEAL_METHOD)
    deal_results = canonicalize_deal_results(deal_results)
    deal_results.to_csv(DEAL_RESULTS_PATH, index=False)
    print("Saved:", DEAL_RESULTS_PATH)

# Write the validated canonical order even when a compatible cache was reused.
deal_results.to_csv(DEAL_RESULTS_PATH, index=False)
print("Exact sample identity verified for all 420 DEAL rows.")

gc.collect()
if device.type == "mps":
    torch.mps.empty_cache()


Evaluating official DEAL on all 420 paired benchmark samples.
DEAL sample   1/420 | elapsed 0.0 min | estimated remaining 0.4 min
DEAL sample  10/420 | elapsed 1.4 min | estimated remaining 62.6 min
DEAL sample  20/420 | elapsed 2.9 min | estimated remaining 60.2 min
DEAL sample  30/420 | elapsed 4.3 min | estimated remaining 58.3 min
DEAL sample  40/420 | elapsed 5.9 min | estimated remaining 57.7 min
DEAL sample  50/420 | elapsed 7.4 min | estimated remaining 55.7 min
DEAL sample  60/420 | elapsed 8.9 min | estimated remaining 54.3 min
DEAL sample  70/420 | elapsed 10.4 min | estimated remaining 52.9 min
DEAL sample  80/420 | elapsed 11.9 min | estimated remaining 51.3 min
DEAL sample  90/420 | elapsed 13.5 min | estimated remaining 49.9 min
DEAL sample 100/420 | elapsed 14.9 min | estimated remaining 48.2 min
DEAL sample 110/420 | elapsed 17.1 min | estimated remaining 48.7 min
DEAL sample 120/420 | elapsed 19.6 min | estimated remaining 49.5 min
DEAL sample 130/420 | elapsed 22.2 m

## Summary metrics and solver diagnostics

The per-image file remains the authoritative result. The compact DIV2K
table is convenient for importing into the combined benchmark, while
the solver table reports the adaptive effort used by official DEAL.


In [7]:
div2k_summary = method_summary(
    deal_results.loc[deal_results["dataset"] == "div2k"].copy()
)
div2k_summary.to_csv(OUTPUT_DIR / "method_summary_div2k.csv", index=False)

diagnostic_prefixes = (
    "solver_", "deal_", "attention_mean_", "effective_weight_"
)
solver_columns = ["sample_id", "dataset", "image_name", "sigma"] + [
    column
    for column in deal_results.columns
    if column.startswith(diagnostic_prefixes)
]
deal_solver_diagnostics = deal_results[solver_columns].copy()
deal_solver_diagnostics.to_csv(
    OUTPUT_DIR / "deal_solver_diagnostics.csv", index=False
)

display(div2k_summary)
if {
    "solver_outer_iterations", "solver_total_inner_iterations"
}.issubset(deal_results):
    display(
        deal_results.groupby(["dataset", "sigma"])[
            ["solver_outer_iterations", "solver_total_inner_iterations"]
        ].agg(["mean", "std", "min", "max"])
    )


,dataset,sigma,method,images,mse_mean,mse_std,mse_median,psnr_mean,psnr_std,psnr_median,ssim_mean,ssim_std,ssim_median
0,div2k,0.05,DEAL (official pretrained),100,0.000713,0.000293,0.000705,31.895141,2.058904,31.520550,0.947924,0.039595,0.956018
1,div2k,0.10,DEAL (official pretrained),100,0.001734,0.000715,0.001646,28.013940,1.970307,27.836781,0.887200,0.064377,0.898218
2,div2k,0.15,DEAL (official pretrained),100,0.002901,0.001155,0.002736,25.736168,1.841773,25.628905,0.826102,0.083061,0.842115
3,div2k,0.20,DEAL (official pretrained),100,0.004292,0.001666,0.004084,24.011281,1.778704,23.891392,0.764278,0.098652,0.778893


solver_outer_iterations                      \
                                  mean        std min  max   
dataset  sigma                                               
div2k    0.05                    32.49   8.064757  16   63   
         0.10                    52.20  17.895319  22  125   
         0.15                    65.62  19.254164  29  136   
         0.20                    71.79  20.177142  29  154   
geometry 0.05                    11.60   2.792848   9   15   
         0.10                    15.00   5.244044   9   21   
         0.15                    14.60   5.272571   8   20   
         0.20                    14.20   4.438468   9   21   

               solver_total_inner_iterations                         
                                        mean         std  min   max  
dataset  sigma                                                       
div2k    0.05                         256.29   43.992951  145   388  
         0.10                         444.92   89.654407  244   838  
         0.15                         604.12  107.380686  344   926  
         0.20                         711.02  129.040983  415  1088  
geometry 0.05                         105.80   30.760364   73   140  
         0.10                         155.20   44.516289  102   213  
         0.15                         179.00   38.820098  122   214  
         0.20                         205.40   34.268061  148   237

## Inference-time measurement

Runtime is measured on three DIV2K samples at $\sigma=0.10$, batch size
one, after one warm-up, with three timed repetitions and device
synchronisation. These settings match the disclosed comparison protocol
used by the main benchmark notebook.


In [8]:
sigma_index = SIGMAS.index(TIMING_SIGMA)
natural_count = len(div2k_clean)
timing_indices = range(
    sigma_index * natural_count,
    sigma_index * natural_count + natural_count,
)
timing_dataset = Subset(div2k_benchmark, list(timing_indices))
timing_loader = DataLoader(
    timing_dataset,
    batch_size=1,
    shuffle=False,
    num_workers=0,
)

timing_detail = benchmark_inference_runtime(
    deal_model,
    timing_loader,
    device,
    max_batches=TIMING_MAX_BATCHES,
    warmup_runs=TIMING_WARMUP_RUNS,
    repeats=TIMING_REPEATS,
)
timing_detail.insert(0, "method", DEAL_METHOD)
timing_summary = pd.DataFrame(
    [
        runtime_summary(
            timing_detail,
            method=DEAL_METHOD,
            model=deal_model,
            device=device,
        )
    ]
)

timing_detail.to_csv(OUTPUT_DIR / "inference_timing_detail.csv", index=False)
timing_summary.to_csv(OUTPUT_DIR / "inference_time_comparison.csv", index=False)
display(timing_summary)

gc.collect()
if device.type == "mps":
    torch.mps.empty_cache()


,method,device,device_description,parameter_count,timed_measurements,timed_images_per_repeat,mean_runtime_ms_per_image,std_runtime_ms_per_image,median_runtime_ms_per_image,images_per_second,solver_iterations,solver_protocol,outer_iteration_cap,inner_iteration_cap,inner_tolerance,outer_tolerance,mean_outer_iterations,mean_last_inner_iterations,mean_total_inner_iterations
0,DEAL (official pretrained),mps,Apple Metal Performance Shaders (MPS),468570,9,3,11341.68112,2503.915266,10832.595417,0.08817,None,official_notebook,1000,1000,0.000001,0.00001,35.0,1.0,344.0


## Output audit

This final cell records the files produced by the notebook. It also
provides a compact check before the DEAL CSV is consumed by Notebook 6b.


In [9]:
expected_outputs = [
    OUTPUT_DIR / "benchmark_manifest.csv",
    OUTPUT_DIR / "model_configuration.csv",
    DEAL_RESULTS_PATH,
    OUTPUT_DIR / "method_summary_div2k.csv",
    OUTPUT_DIR / "deal_solver_diagnostics.csv",
    OUTPUT_DIR / "inference_timing_detail.csv",
    OUTPUT_DIR / "inference_time_comparison.csv",
]

output_audit = pd.DataFrame(
    {
        "file": [path.name for path in expected_outputs],
        "path": [str(path) for path in expected_outputs],
        "exists": [path.is_file() for path in expected_outputs],
        "size_bytes": [path.stat().st_size if path.is_file() else np.nan for path in expected_outputs],
    }
)
output_audit.to_csv(OUTPUT_DIR / "output_audit.csv", index=False)
display(output_audit)

missing_outputs = output_audit.loc[~output_audit["exists"], "file"].tolist()
if missing_outputs:
    raise RuntimeError(f"The DEAL benchmark is missing outputs: {missing_outputs}")

print("Official DEAL benchmark complete.")
print("Import into Notebook 6 from:", DEAL_RESULTS_PATH)


,file,path,exists,size_bytes
0,benchmark_manifest.csv,/Users/kyrillguba/Documents/Data Analytics MSc...,True,129278
1,model_configuration.csv,/Users/kyrillguba/Documents/Data Analytics MSc...,True,602
2,per_image_metrics_deal_official.csv,/Users/kyrillguba/Documents/Data Analytics MSc...,True,316783
3,method_summary_div2k.csv,/Users/kyrillguba/Documents/Data Analytics MSc...,True,1060
4,deal_solver_diagnostics.csv,/Users/kyrillguba/Documents/Data Analytics MSc...,True,152287
5,inference_timing_detail.csv,/Users/kyrillguba/Documents/Data Analytics MSc...,True,845
6,inference_time_comparison.csv,/Users/kyrillguba/Documents/Data Analytics MSc...,True,585


Official DEAL benchmark complete.
Import into Notebook 6 from: /Users/kyrillguba/Documents/Data Analytics MSc/Thesis/Python/cleaned_pipeline/results/deal_official_benchmark/per_image_metrics_deal_official.csv
